In [1]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.signal as sig
from scipy.fft import dct
from sklearn.mixture import GaussianMixture

plt.rcParams['figure.figsize'] = (10, 4)
np.set_printoptions(precision=3, suppress=True)

In [2]:
# La idea es que utilicen estas funciones, diccionarios, listas y variables sin modificar sus valores
fs = 16000

formantes = {
    'a': [750, 1300, 2500],
    'i': [300, 2300, 3000],
    'u': [350, 800, 2300],
}

anchos_BW = [80, 90, 120]

def sintetizar_vocal(formant_freqs, fs=16000, F0=120, duracion=0.6, bws=anchos_BW):
    N_x = int(duracion * fs)
    fuente = np.zeros(N_x)
    periodo = int(fs / F0)
    fuente[::periodo] = 1.0
    salida = fuente.copy()
    for f, bw in zip(formant_freqs, bws):
        r = np.exp(-np.pi * bw / fs)
        theta = 2 * np.pi * f / fs
        b = [1 - r*r]
        a = [1, -2*r*np.cos(theta), r*r]
        salida = sig.lfilter(b, a, salida)
    return salida / np.max(np.abs(salida))

def hz_to_mel(f):
    return 2595 * np.log10(1 + f / 700)

def mel_to_hz(m):
    return 700 * (10**(m / 2595) - 1)

def enmarcar(senial, fs, win_ms=25, hop_ms=10):
    L = int(round(win_ms * 1e-3 * fs))
    H = int(round(hop_ms * 1e-3 * fs))
    n_frames = 1 + (len(senial) - L) // H
    frames = np.stack([senial[i*H:i*H+L] for i in range(n_frames)])
    return frames * np.hamming(L)

def banco_mel(n_mels, n_fft, fs, f_min=80, f_max=None):
    if f_max is None:
        f_max = fs / 2
    puntos_mel = np.linspace(hz_to_mel(f_min), hz_to_mel(f_max), n_mels + 2)
    puntos_hz = mel_to_hz(puntos_mel)
    bins = np.floor(n_fft * puntos_hz / fs).astype(int)
    bins = np.clip(bins, 0, n_fft // 2)
    B = np.zeros((n_mels, n_fft//2 + 1))
    for m in range(1, n_mels + 1):
        izq, centro, der = bins[m-1], bins[m], bins[m+1]
        for k in range(izq, centro):
            B[m-1, k] = (k - izq) / max(centro - izq, 1)
        for k in range(centro, der):
            B[m-1, k] = (der - k) / max(der - centro, 1)
    return B

def mfcc_propio(senial, fs=16000, n_mfcc=13, n_mels=26, n_fft=512):
    senial_pe = np.append(senial[0], senial[1:] - 0.97 * senial[:-1])
    frames = enmarcar(senial_pe, fs)
    X = np.fft.rfft(frames, n=n_fft, axis=1)
    S = np.abs(X)**2
    B = banco_mel(n_mels, n_fft, fs)
    E = S @ B.T
    logE = np.log(E + 1e-10)
    return dct(logE, type=2, axis=1, norm='ortho')[:, :n_mfcc]

# Ejercicio 3

In [ ]:
audio_a_ref = sintetizar_vocal(formantes[a], F0 = 120) 

display(Audio(audio, rate=fs))
audio_a_nivel = None ### <-- COMPLETAR
audio_a_pitch = None ### <-- COMPLETAR
audio_i_ref = None ### <-- COMPLETAR

seniales = {
    'a_ref': audio_a_ref, # Señal A
    'a_nivel': audio_a_nivel, # Señal B
    'a_pitch': audio_a_pitch, # Señal C
    'i_ref': audio_i_ref, # Señal D
}

matrices_mfcc = None ### <-- COMPLETAR
promedios_temporales = None ### <-- COMPLETAR

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
for nombre, vec_medio in promedios_temporales.items():
    ax.plot(np.arange(len(vec_medio)), vec_medio, marker='o', label=nombre)
ax.set_xlabel(...) ### <-- COMPLETAR
ax.set_ylabel('MFCC medio')
ax.set_title('MFCC medios de las cuatro señales')
ax.legend()
ax.grid(alpha=0.3)
plt.show()

In [ ]:
def distancia_mfcc(a, b, incluir_c0=True):
    return None ### <-- COMPLETAR FUNCIÓN

referencia = promedios_temporales['a_ref']

for nombre in ['a_nivel', 'a_pitch', 'i_ref']:
    d_13 = None ### <-- COMPLETAR
    d_12 = None ### <-- COMPLETAR
    print(f'Para {nombre} obtenemos: {d_13:.2f} (con c0) y {d_12:.2f} (sin c0)')

# Ejercicio 4

In [ ]:
audio_a_100 = None ### <-- COMPLETAR (señal de 100 Hz para /a/)
audio_a_200 = None ### <-- COMPLETAR (señal de 200 Hz para /a/)
audio_a_100_bajo = None ### <-- COMPLETAR (señal adicional)
audio_i_100 = None ### <-- COMPLETAR (señal adicional)

mfcc_a_100 = None ### <-- COMPLETAR
mfcc_a_200 = None ### <-- COMPLETAR
mfcc_a_100_bajo = None ### <-- COMPLETAR
mfcc_i_100 = None ### <-- COMPLETAR

# 13 y 12 corresponden a la cantidad de MFCC considerados
X_entrenamiento_13 = None ### <-- COMPLETAR
X_entrenamiento_12 = None ### <-- COMPLETAR

In [ ]:
def crear_gmm():
    ### COMPLETAR LO QUE FALTA
    return GaussianMixture(n_components=..., covariance_type=..., init_params='random_from_data', n_init=5, max_iter=300, reg_covar=1e-3, random_state=0) 

modelo_13 = crear_gmm()
modelo_13.fit(None) ### <-- COMPLETAR

modelo_12 = crear_gmm()
modelo_12.fit(None) ### <-- COMPLETAR

for nombre, C in {'a_100': mfcc_a_100, 'a_200': mfcc_a_200}.items():
    responsabilidades = None ### <-- COMPLETAR
    responsabilidades_medias = None ### <-- COMPLETAR
    print(f'{nombre}: {responsabilidades_medias}')

In [ ]:
def log_verosimilitud_media(modelo, C, incluir_c0=True):
    return None ### <-- COMPLETAR FUNCIÓN

casos = {
    'a_100': mfcc_a_100,
    'a_200': mfcc_a_200,
    'a_100_bajo': mfcc_a_100_bajo,
    'i_100': mfcc_i_100,
}

for nombre, C in casos.items():
    ell_13 = None ### <-- COMPLETAR
    ell_12 = None ### <-- COMPLETAR
    print(f'Para {nombre} obtenemos: {ell_13:.2f} (con c0)  ¿+{ell_12:.2f} (sin c0)')